# 🕸️ Modern Web Scraping 🕸️

Welcome to the next generation of web scraping! As of **2026**, the industry has moved from manually parsing HTML to using **AI-Native** tools that "understand" web pages like a human does.

### Why the change?
1. **Dynamic Content:** Sites are now mostly JavaScript-based, making simple `requests` ineffective.
2. **Anti-Bot Tech:** Protections like Cloudflare and DataDome have become extremely sophisticated.
3. **AI Demand:** We now scrape data primarily to feed **LLMs** and **RAG** systems, which require clean Markdown/JSON rather than raw HTML.

## 🧠 Modern Warm-up Exercise
**Match the modern scraping concepts with their correct descriptions:**

<table border="1">
    <tr>
        <th>Concept</th>
        <th>Description</th>
    </tr>
    <tr>
        <td>Agentic Scraping</td>
        <td>Using an AI to navigate websites, click buttons, and solve problems autonomously.</td>
    </tr>
    <tr>
        <td>Crawl4AI</td>
        <td>A popular open-source library specifically built for high-speed AI data extraction.</td>
    </tr>
    <tr>
        <td>Firecrawl</td>
        <td>An API that crawls an entire domain and outputs structured JSON without manual selectors.</td>
    </tr>
    <tr>
        <td>Managed Browser</td>
        <td>A cloud-based browser API that handles proxies and anti-bot bypassing for you.</td>
    </tr>
    <tr>
        <td>Semantic Extraction</td>
        <td>The process of converting complex HTML into LLM-readable Markdown automatically.</td>
    </tr>
</table>

## 🛠️ Step 1: Setup

We load the API keys from the `.env` file and initialize our clients.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from firecrawl import Firecrawl

# Load API keys from .env file
load_dotenv()

# Initialize the Firecrawl client (v4.x API)
app = Firecrawl(api_key=os.getenv('FIRECRAWL_API_KEY'))



ModuleNotFoundError: No module named 'dotenv'

## 🚀 Step 2: Scrape a Page to Markdown

The biggest shift in 2026 is that we no longer parse raw HTML. Instead, Firecrawl converts the whole page into **clean Markdown** ready for an LLM.

In [7]:
# Scrape a page to clean Markdown
result = app.scrape('https://news.ycombinator.com', formats=['markdown'])

print('--- Scraped Markdown (First 500 chars) ---')
print(result.markdown[:500])

--- Scraped Markdown (First 500 chars) ---
|     |     |     |
| --- | --- | --- |
| |     |     |     |
| --- | --- | --- |
| [![](https://news.ycombinator.com/y18.svg)](https://news.ycombinator.com/) | **[Hacker News](https://news.ycombinator.com/news)** [new](https://news.ycombinator.com/newest) \| [past](https://news.ycombinator.com/front) \| [comments](https://news.ycombinator.com/newcomments) \| [ask](https://news.ycombinator.com/ask) \| [show](https://news.ycombinator.com/show) \| [jobs](https://news.ycombinator.com/jobs) \| [subm


## 📊 Step 3: Structured Extraction & Save to CSV

Now the real magic: we describe what we want in plain English and Firecrawl returns structured JSON — no CSS selectors needed.

**Requires:** `FIRECRAWL_API_KEY` only ✅

> 💡 **Tip:** You can use just a `prompt` — no `schema` needed!
> Adding a `schema` gives you control over the exact output key names,
> but for most use cases the `prompt` alone is enough.

In [8]:
# Use app.extract() for AI-powered structured data extraction
# A plain-English prompt is all you need — no CSS selectors!
response = app.extract(
    ['https://news.ycombinator.com'],
    prompt='Extract the top 4 news stories. For each one, get the title and the number of points.',
        schema={
        'type': 'object',
        'properties': {
            'items': {
                'type': 'array',
                'items': {
                    'type': 'object',
                    'properties': {
                        'title': {'type': 'string'},
                        'points': {'type': 'integer'}
                    }
                }
            }
        }
    }
)


# response.data is a dict — grab the first (and only) list it contains
# This works regardless of what key name Firecrawl chooses
items = list(response.data.values())[0]

# Convert to DataFrame and save to CSV
df = pd.DataFrame(items)
df.to_csv('hacker_news_firecrawl.csv', index=False)

print('✅ Success! Data saved to hacker_news_firecrawl.csv')
print(f'Credits used: {response.credits_used}')
print()
print('--- Extracted Data ---')
print(df)

✅ Success! Data saved to hacker_news_firecrawl.csv
Credits used: 25

--- Extracted Data ---
                                               title  points
0                    Talking to strangers at the gym    1264
1      GameStop makes $55.5B takeover offer for eBay     659
2                             I am worried about Bun     449
3  Microsoft Edge stores all passwords in memory ...     482


---
## 🤖 Bonus Step: Agentic Scraping with OpenAI

> ⚠️ **Requires:** `OPENAI_API_KEY` with active credits

This is the next level beyond Firecrawl. Instead of telling the AI *what to extract*, we give an **autonomous agent** a task in plain English — and it navigates the browser itself, clicks, scrolls, and saves results, just like a human would.

**Use case:** Tasks that require login, multi-step navigation, or dynamic interactions that Firecrawl can't handle.

In [ ]:
# ⚠️ Only run this cell if your OpenAI account has active credits!
from browser_use.llm import ChatOpenAI
from browser_use import Agent

async def run_agent_task():
    # Use browser_use's own ChatOpenAI wrapper (not langchain_openai)
    llm = ChatOpenAI(model='gpt-4o')

    agent = Agent(
        task=(
            "Go to https://news.ycombinator.com, "
            "extract the top 5 news titles and their points, "
            "and save them to a file named 'hacker_news_agent.csv'."
        ),
        llm=llm
    )

    print('🚀 Agent started — navigating the browser autonomously...')
    await agent.run()
    print('✅ Agent task complete!')

    if os.path.exists('hacker_news_agent.csv'):
        df_agent = pd.read_csv('hacker_news_agent.csv')
        print('\n--- Agent-generated CSV ---')
        print(df_agent)
    else:
        print('CSV not found. Check the agent output above for details.')

# Run the agent
await run_agent_task()

## 🛡️ Modern Best Practices

- **Respect `robots.txt` for AI:** Many sites now have a separate section for AI crawlers. Always check it!
- **Use `extract()` over `scrape()` for structured data:** It is cheaper and more accurate.
- **Prefer Markdown:** Markdown is the native language of LLMs. Use `formats=['markdown']` for RAG pipelines.
- **Audit Your Data:** With the **EU AI Act**, keep a record of where your training data came from.


**Next Steps:**
- Try pointing `extract()` at a different website with a custom prompt.
- Feed the Markdown into a RAG pipeline using `langchain`.
